# MIA Visualization

In [1]:
import os
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
train_steps_dcr = [0.66060, 0.67230, 0.68049, 0.67850, 0.70130, 0.71829, 0.74869, 0.76339, 0.77380, 0.77319]
train_steps_bb = [0.10800, 0.104, 0.115, 0.107, 0.135, 0.158, 0.180, 0.210, 0.230, 0.230]
train_steps = [5e03, 1e04, 1.5e04, 2.5e04, 5e04, 1e05, 2e05, 3e05, 4e05, 5e05]

diffusion_steps_dcr = [0.71149, 0.73510, 0.76400, 0.77149, 0.70600, 0.71430, 0.71000, 0.74450, 0.70990, 0.70969]
diffusion_steps_bb = [0.1145, 0.1345, 0.1825, 0.1555, 0.1615, 0.1795, 0.1855, 0.1795, 0.178, 0.187]
diffusion_steps = [10, 20, 50, 80, 100, 500, 1000, 2000, 3000, 4000]

synthetic_size = ["1x", "5x", "10x"]
bb_10k = [0.283, 0.365, 0.341]
bb_20k = [0.215, 0.273, 0.280]
dcr_20k = [0.749, 0.748, 0.749]

batch_size_dcr = [0.69 , 0.71 , 0.70]
batch_size_bb = [0.14, 0.1795, 0.1515]
batch_size = [2048, 4096, 8192]

train_size = [1e04, 2e04, 5e04]
train_size_delta_dcr = [0.18000, 0.08800, 0.00800]
train_size_bb = [0.2265, 0.215, 0.120]
train_size_dcr = [0.68800, 0.74869, 0.83876]
train_size_ideal_dcr = [0.50000, 0.66666, 0.83333]

model_variation = ["Narrow<br>Shallow<br>Short", "Narrow<br>Shallow", "Narrow", "Default", "Wide<br>Deep", "Wide<br>Deep<br>Long"]
mia_ensemble_model_variation = [0.128, 0.138, 0.172, 0.215, 0.198, 0.217]
dcr = [0.7035, 0.7375, 0.6984, 0.7400, 0.7137, 0.7143]
dcr_ideal = [2/3, 2/3, 2/3, 2/3, 2/3, 2/3]


In [3]:
FIG_HEIGHT = 800
FIG_WIDTH = 1200
FONT_SIZE = 44

## Utilities

In [4]:
def format_metric_name(metric_name: str) -> str:
    """
    Prettify diagram texts by replacing
    snake case with regular title case.
    """
    output_words = []
    for word in metric_name.split("_"):
        word = word.title() if word not in ["FPR", "TPR"] else word.upper()
        output_words.append(word)

    return " ".join(output_words)

## Plotting

In [5]:
repo_abs_path = Path(os.path.abspath("")).parent.parent

plots_dir = f"{repo_abs_path}/examples/visualizations/berka_ensemble_training"

In [6]:
def customize_figure_layout(gen_fig: Any, width = FIG_WIDTH, height = FIG_HEIGHT) -> None:
    gen_fig.update_xaxes(automargin=True)
    gen_fig.update_yaxes(automargin=True)

    gen_fig.update_layout(
        height=height,
        width=width,
        font_color="black",
        legend=dict(
            y=1.0,
            x=0.5,
            xanchor="center",
            yanchor="bottom",
            orientation="h",
            valign="top",
            title_text="",
            font=dict(size=FONT_SIZE-4),
            title_font_family="Helvetica",
        ),
        title_font_family="Helvetica",
        title_x=0.5,
        title_y=0.99,
        margin=dict(l=0, r=0, t=40, b=0, pad=0),
        plot_bgcolor="white",
        font=dict(size=FONT_SIZE, family="Helvetica"),
    )

def trim_png_whitespace(image_path: str, pad: int = 2) -> None:
    """Crop near-white borders from a saved PNG."""
    import numpy as np
    from PIL import Image

    img = Image.open(image_path)
    arr = np.asarray(img)
    content = np.any(arr[:, :, :3] < 250, axis=2) if arr.ndim == 3 else arr < 250
    rows = np.any(content, axis=1)
    cols = np.any(content, axis=0)
    if not rows.any() or not cols.any():
        return

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    rmin = max(0, int(rmin) - pad)
    cmin = max(0, int(cmin) - pad)
    rmax = min(arr.shape[0] - 1, int(rmax) + pad)
    cmax = min(arr.shape[1] - 1, int(cmax) + pad)
    img.crop((cmin, rmin, cmax + 1, rmax + 1)).save(image_path)


def save_and_display_figure(gen_fig: Any, template_name: str, plots_dir: str) -> None:
    plot_png_path = f"{plots_dir}/{template_name}.png"
    plot_pdf_path = f"{plots_dir}/{template_name}.pdf"

    os.makedirs(plots_dir, exist_ok=True)
    gen_fig.write_image(plot_png_path, scale=2)
    gen_fig.write_image(plot_pdf_path)
    trim_png_whitespace(plot_png_path)

    display(gen_fig)

In [7]:
df = {"Train Steps": train_steps, "Ensemble (BB)": train_steps_bb, "DCR": train_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Train Steps",
    y=["Ensemble (BB)", "DCR"],
    markers=True,
)
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=2/3, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[5e03, 1e04, 2e04, 5e04, 1e05, 2e05, 5e05],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "berka_ensemble_dcr_train_steps", plots_dir)

In [8]:
df = {"Diffusion Steps": diffusion_steps, "Ensemble (BB)": diffusion_steps_bb, "DCR": diffusion_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Diffusion Steps",
    y=["Ensemble (BB)", "DCR"],
    markers=True,
)
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=2/3, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[1e01, 2e01, 5e01, 1e02, 2e02, 5e02, 1e03, 2e03, 5e03],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "berka_ensemble_dcr_diffusion_steps", plots_dir)

In [9]:
df = {"Synthetic Size": synthetic_size, "Ensemble (BB) 10K": bb_10k, "Ensemble (BB) 20K": bb_20k, "DCR 20K": dcr_20k}

fig = px.line(
    data_frame=df,
    x="Synthetic Size",
    y=["Ensemble (BB) 10K", "Ensemble (BB) 20K", "DCR 20K"],
    markers=True,
)
fig.data[2].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=2/3, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=["1x", "5x", "10x"],
    ),
)
fig.update_xaxes(
   ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "berka_ensemble_dcr_synthetic_size", plots_dir)

In [10]:
df = {"Batch Size": batch_size, "Ensemble (BB)": batch_size_bb, "DCR": batch_size_dcr}

fig = px.line(
    data_frame=df,
    x="Batch Size",
    y=["Ensemble (BB)", "DCR"],
    markers=True,
)
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=2/3, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=["2048", "4096", "8192"],
    ),
)
fig.update_xaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "berka_ensemble_dcr_batch_size", plots_dir)

In [11]:
df = {"Train Size": train_size, "Ensemble (BB)": train_size_bb, "DCR": train_size_dcr, "Ideal DCR": train_size_ideal_dcr}

fig = px.line(
    data_frame=df,
    x="Train Size",
    y=["Ensemble (BB)", "DCR"],
    markers=True,
)
fig.add_trace(go.Scatter(x=df["Train Size"], y=df["Ideal DCR"], mode='lines', name='Ideal DCR'))
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.data[2].line.color = "#660066"
fig.data[2].line.dash = "dash"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=train_size,
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E",
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "berka_ensemble_dcr_train_size", plots_dir)

In [12]:
df = {"Model Variations": model_variation, "Ensemble (BB)": mia_ensemble_model_variation, "DCR": dcr}

fig = px.line(
    data_frame=df,
    x="Model Variations",
    y=["Ensemble (BB)", "DCR"],
    markers=True,
)
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=2/3, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=model_variation,
    ),
)
fig.update_xaxes(
   ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "berka_ensemble_model_variation", plots_dir)